# Data Versioning & Data Lineage

In this lab, you version data with [DVC](https://dvc.org/) and keep track of the processing that produces them. DVC separates data versioning, git tracking their metadata (`.dvc` files), from data storage, here [DagsHub](https://dagshub.com/)'s, which also hosts your git repository.

The commands are those of the course's “End-to-end workflow” frame. A line starting with `!` is a shell command, where a Python variable goes between braces (`{token}`). The steps follow on from each other: each one starts from the previous one's repository.

## Setup — Please run the cell

This cell installs DVC and its S3 extension, then moves you into the project's directory, `~/dvc-project`. Run it again if the runtime restarts.

In [ ]:
!apt install tree
!pip install dvc dvc-s3
!mkdir -p ~/dvc-project
%cd ~/dvc-project

## 1. Create the git repository

Create an account on [DagsHub](https://dagshub.com/), then a blank git repository. When you have created the repository, click on “*Get started with Data*” and get your secret token from the “*Connection credentials*” section.

Fill the `owner`, `repo` and `token` variables with your DagsHub username, the repository's name and your secret token, then run the following cell: it creates the project's git repository, linked to your DagsHub repository.

In [ ]:
owner = ""
repo = ""
token = ""

In [ ]:
!git init -b main
!git config user.name "Jeanne Durant"
!git config user.email "jeanne@durant.fr"
!git remote add origin https://{token}@dagshub.com/{owner}/{repo}.git

## 2. Set up DVC and its remote storage

Initialize DVC in the repository, then declare your DagsHub repository's storage, which is S3-compatible, as DVC's **default** remote storage, with its server's address and your credentials (`dvc remote modify`). DagsHub gives these commands in the “*Data*” tab of your repository's green “*Remote*” button: add the option that makes this storage the default one. Then commit DVC's configuration (the `.dvc` directory) and push it to DagsHub.

Print `.dvc/config` and `.dvc/config.local`: which one does git track, and why are the credentials in the other one?

**Useful documentation**

- https://dvc.org/doc/command-reference/remote/modify
- https://dvc.org/doc/user-guide/data-management/remote-storage/amazon-s3

In [ ]:
# Your code here

### Additional hints

Name the storage `origin`, with the URL `s3://dvc`; `-d` makes it the default storage, the one `dvc push` and `dvc pull` use without `-r`. The server's address, `endpointurl`, is `https://dagshub.com/{owner}/{repo}.s3`. Both credentials, `access_key_id` and `secret_access_key`, are your token, and the `--local` option writes them to `.dvc/config.local`, which the `.dvc/.gitignore` that `dvc init` creates keeps out of git.

```python
!dvc init
!dvc remote add -d origin s3://dvc
!dvc remote modify origin endpointurl https://dagshub.com/{owner}/{repo}.s3
!dvc remote modify origin --local access_key_id {token}
!dvc remote modify origin ... secret_access_key {token}
!cat .dvc/config .dvc/config.local
!git add .dvc
!git commit -m "Initialize DVC"
!git push ...
```

### Solution

In [ ]:
!dvc init
!dvc remote add -d origin s3://dvc
!dvc remote modify origin endpointurl https://dagshub.com/{owner}/{repo}.s3
!dvc remote modify origin --local access_key_id {token}
!dvc remote modify origin --local secret_access_key {token}

In [ ]:
!cat .dvc/config .dvc/config.local

In [ ]:
!git add .dvc
!git commit -m "Initialize DVC"
!git push origin main

git tracks `.dvc/config`, which describes the storage for the whole team. `.dvc/config.local`, which `.dvc/.gitignore` excludes, keeps the credentials on this machine: they never go into git, and everyone puts their own there (step 8).

## 3. Retrieve a first version of the data

Retrieve with `dvc get` the `wikipedia-movie-plots/wiki_movie_plots_deduped.csv` file of the `https://github.com/shuuchuu/datasets` repository, as `data.csv`, then have DVC track it with `dvc add`.

Then print `data.csv.dvc` (`!cat data.csv.dvc`) and DVC's cache (`!tree .dvc/cache`): what does the `.dvc` file hold, and under which name does DVC store the data in its cache?

**Useful documentation**

- https://dvc.org/doc/command-reference/get

In [ ]:
# Your code here

### Additional hints

`dvc get <repository> <path in the repository> -o <file>` downloads a file from a git or DVC repository, without tracking it. `dvc add <file>` copies it into the cache, writes `<file>.dvc` and adds the file to `.gitignore`.

```python
!dvc get https://github.com/shuuchuu/datasets ... -o data.csv
!dvc add ...
!cat data.csv.dvc
!tree .dvc/cache
```

### Solution

In [ ]:
!dvc get https://github.com/shuuchuu/datasets wikipedia-movie-plots/wiki_movie_plots_deduped.csv -o data.csv

In [ ]:
!dvc add data.csv

In [ ]:
!cat data.csv.dvc
!tree .dvc/cache

`data.csv.dvc` only holds the file's md5, its size and its path: this small file is what git versions. The cache stores the data under their md5 (`files/md5/<first 2 characters>/<the rest>`): DVC finds them from the `.dvc` file.

## 4. Save this version

Commit the files `dvc add` told you to add to git, tag this commit `v1`, and push the branch and the tag to DagsHub. Finally, send the data to the remote storage with `dvc push`.

What does `git push` send, and what does `dvc push` send?

In [ ]:
# Your code here

### Additional hints

`dvc add` printed the files to add to git: `data.csv.dvc` and `.gitignore`. `git push origin main v1` pushes the branch and the tag in one command, and `dvc push`, without `-r`, sends to the default storage.

```python
!git add ... .gitignore
!git commit -m "Raw data"
!git tag ...
!git push origin main ...
!dvc push
```

### Solution

In [ ]:
!git add data.csv.dvc .gitignore
!git commit -m "Raw data"
!git tag v1

In [ ]:
!git push origin main v1
!dvc push

`git push` only sends the metadata: `data.csv.dvc` and `.gitignore`, a few lines. `dvc push` sends the data file itself (81 MB), from the cache, to the storage.

## 5. Modify the data

The following cell modifies the data: it sorts the lines of `data.csv` in reverse order. Run it, then save this new version like the first one, with the `v2` tag, and send it to DagsHub (git and DVC).

Compare `data.csv.dvc` and DVC's cache to what they were at step 3: what changed?

In [ ]:
!sort -r data.csv -o data.csv

In [ ]:
# Your code here

### Additional hints

The commands of steps 3 and 4: `dvc add` computes the md5 again and puts the new version in the cache, `git add` saves the new `data.csv.dvc` (`.gitignore` didn't change).

```python
!dvc add data.csv
!cat data.csv.dvc
!tree .dvc/cache
!git add ...
!git commit -m "Data v2"
!git tag ...
!git push origin main ...
!dvc push
```

### Solution

In [ ]:
!dvc add data.csv

In [ ]:
!cat data.csv.dvc
!tree .dvc/cache

In [ ]:
!git add data.csv.dvc
!git commit -m "Data v2"
!git tag v2

In [ ]:
!git push origin main v2
!dvc push

The md5 in `data.csv.dvc` changed, not the size: the same lines, in another order. The cache now holds both versions, each under its md5: going back to the first one doesn't need to download it again.

## 6. Back to the original data

You find a problem in the new data: go back to the first version. Restore with `git restore` the `data.csv.dvc` of the `v1` tag, then, with `dvc checkout`, the data file it points to. Commit this rollback, with the `v3` tag, and send it to DagsHub.

Why does `dvc push` have nothing to send this time?

In [ ]:
# Your code here

### Additional hints

`git restore --source v1 <file>` puts the file back as it was at `v1`, without switching branches. `dvc checkout` then puts back into the working directory, from the cache, the data the `.dvc` files point to.

```python
!git restore --source ... data.csv.dvc
!dvc checkout
!cat data.csv.dvc
!git add data.csv.dvc
!git commit -m "Back to data v1"
!git tag ...
!git push origin main ...
!dvc push
```

### Solution

In [ ]:
!git restore --source v1 data.csv.dvc
!dvc checkout

In [ ]:
!cat data.csv.dvc

In [ ]:
!git add data.csv.dvc
!git commit -m "Back to data v1"
!git tag v3

In [ ]:
!git push origin main v3
!dvc push

The `v1` data are already on the storage, under the same md5: DVC only sends what the storage lacks. The git history keeps the three versions, and `v2` can be retrieved the same way.

## 7. A pipeline stage

The following cell writes the `upper.py` script, which copies the file given as first argument into the one given as second argument, in uppercase. Run it.

Add with `dvc stage add` an `uppercase` stage that produces `data-upper.csv` from `data.csv` with this script, declaring the script and the data as dependencies. Run it with `dvc repro`, then save the script and the pipeline in git, with the `dvc.lock` file `dvc repro` writes, and send its output to the storage.

Then run `dvc repro` again: what does it do, and why? What does `dvc.lock` hold?

In [ ]:
%%writefile upper.py
from pathlib import Path
from sys import argv

text = Path(argv[1]).read_text(encoding="utf-8")
Path(argv[2]).write_text(text.upper(), encoding="utf-8")

In [ ]:
# Your code here

### Additional hints

The options of the course's “Defining a stage” frame: `--name` (the stage's name), `--deps` (once per dependency), `--outs` (the output), then the command. `dvc stage add` writes `dvc.yaml` and adds the output to `.gitignore`; `dvc repro` runs the stage and writes `dvc.lock`.

```python
!dvc stage add --name uppercase --deps ... --deps ... --outs ... python upper.py data.csv data-upper.csv
!dvc repro
!git add upper.py dvc.yaml ... .gitignore
!git commit -m "Uppercase pipeline"
!git push origin main
!dvc push
!dvc repro
!cat dvc.lock
```

### Solution

In [ ]:
!dvc stage add --name uppercase --deps upper.py --deps data.csv --outs data-upper.csv python upper.py data.csv data-upper.csv

In [ ]:
!dvc repro

In [ ]:
!git add upper.py dvc.yaml dvc.lock .gitignore
!git commit -m "Uppercase pipeline"
!git push origin main
!dvc push

In [ ]:
!dvc repro
!cat dvc.lock

The second `dvc repro` skips the stage: it compares the md5 of its dependencies and output to those in `dvc.lock`, and nothing changed. `dvc.lock` records the command and these md5: it says exactly from what and how `data-upper.csv` was produced, which is data lineage.

## 8. On another machine

A colleague joins the project. Clone your DagsHub repository into `~/other-machine`, as she would on her machine, and move into this directory with the `%cd` magic command. The storage's credentials stayed in your `.dvc/config.local`: give them to the clone, then retrieve the data with `dvc pull`.

Does the clone have `data.csv` and `data-upper.csv`? What do `dvc status` and `dvc repro` say?

In [ ]:
# Your code here

### Additional hints

`git clone <url> ~/other-machine`, with the repository URL of step 1; the two `dvc remote modify ... --local` of step 2; `dvc pull` retrieves all the data the `.dvc` and `dvc.lock` files of the current commit point to.

```python
!git clone https://{token}@dagshub.com/{owner}/{repo}.git ...
%cd ~/other-machine
!dvc remote modify origin --local ... {token}
!dvc remote modify origin --local ... {token}
!dvc pull
!ls
!dvc status
!dvc repro
```

### Solution

In [ ]:
!git clone https://{token}@dagshub.com/{owner}/{repo}.git ~/other-machine

In [ ]:
%cd ~/other-machine

In [ ]:
!dvc remote modify origin --local access_key_id {token}
!dvc remote modify origin --local secret_access_key {token}

In [ ]:
!dvc pull

In [ ]:
!ls
!dvc status
!dvc repro

`dvc pull` retrieved `data.csv`, in its `v3` version, and `data-upper.csv`, without running the script again. `dvc status` says the data and the pipeline are up to date, and `dvc repro` has nothing to redo: the clone has exactly the project's data and processing.